In [1]:
import pandas as pd
import numpy as np
import random

def weighted_choice(options):
    r = random.random()
    cumulative = 0
    for value, weight in options:
        cumulative += weight
        if r <= cumulative:
            return value

def rand_range(low, high):
    return random.randint(low, high)

def process_creatures(df):
    # Filter Stage 1
    df = df[df["Stage"] == "Stage 1"].copy()

    # Create new columns
    df["move1_level"] = 1

    df["move2_level"] = df.apply(
        lambda _: weighted_choice([(2, 0.25), (3, 0.50), (4, 0.25)]),
        axis=1
    )
    df["move3_level"] = df.apply(
        lambda _: weighted_choice([(5, 0.25), (6, 0.50), (7, 0.25)]),
        axis=1
    )
    df["move4_level"] = df.apply(
        lambda _: weighted_choice([(8, 0.50), (9, 0.30), (10, 0.20)]),
        axis=1
    )

    # Randomly choose Buff move
    df["buff_move"] = df.apply(
        lambda _: random.choice(["move3"]),
        axis=1
    )

    # Prepare move columns
    df["move1"] = None
    df["move2"] = None
    df["move3"] = None
    df["move4"] = None

    # Assign move powers
    for idx, row in df.iterrows():
        rarity = int(row["Rarity"])

        # Define ranges
        if rarity >= 4:
            ranges = {
                "move1": (30, 45),
                "move2": (40, 65),
                "move3": (50, 75),
                "move4": (60, 90)
            }
        else:
            ranges = {
                "move1": (25, 38),
                "move2": (45, 55),
                "move3": (60, 70),
                "move4": (60, 83)
            }

        # Assign Buff
        buff = row["buff_move"]
        moves = {}

        for m in ["move1", "move2", "move3", "move4"]:
            if m == buff:
                moves[m] = "Buff"
            else:
                low, high = ranges[m]
                moves[m] = rand_range(low, high)

        # Enforce ordering move4 > move3 > move2 > move1
        numeric_moves = {m: v for m, v in moves.items() if v != "Buff"}
        sorted_values = sorted(numeric_moves.values())

        # Reassign sorted values in correct order
        ordered_keys = ["move1", "move2", "move3", "move4"]
        sorted_idx = 0
        for m in ordered_keys:
            if moves[m] != "Buff":
                moves[m] = sorted_values[sorted_idx]
                sorted_idx += 1

        # Save back
        df.at[idx, "move1"] = moves["move1"]
        df.at[idx, "move2"] = moves["move2"]
        df.at[idx, "move3"] = moves["move3"]
        df.at[idx, "move4"] = moves["move4"]

    return df


In [2]:
df = pd.read_csv("Raid Exclusive - Search and Go.csv")
result = process_creatures(df)
print(result)


           id_output              Name    Stage       Type  \
0   Exclusive_01.png         Cryostone  Stage 1     Mystic   
2   Exclusive_03.png           Sushimi  Stage 1     Mystic   
5   Exclusive_06.png  Chargorb Electro  Stage 1       Wind   
6   Exclusive_07.png            Penkit  Stage 1       Wind   
8   Exclusive_09.png         Castanito  Stage 1    Neutral   
11  Exclusive_12.png          Capyfrio  Stage 1    Neutral   
13  Exclusive_14.png       Tulippestus  Stage 1  Celestial   
14  Exclusive_15.png             Tenyz  Stage 1  Celestial   
16  Exclusive_17.png    Chargorb Rusty  Stage 1   Mechanic   
17  Exclusive_18.png         Belliffel  Stage 1   Mechanic   

                   image  Rarity   Evolves to  Evolution candy   HP  Attack  \
0          Cryostone.png     4.0   Ghoulgourd             35.0   80      40   
2            Sushimi.png     3.0     Tunarisu             30.0   40      40   
5   Chargorb Electro.png     4.0          NaN              NaN   62      85   
6

In [3]:
new_df=df.merge(result, left_on="id_output", right_on="id_output", how="left")

In [4]:
new_df["move1 name"] = None 
new_df["move2 name"] = None 
new_df["move3 name"] = None 
new_df["move4 name"] = None 

new_df["move1 buff"] = None 
new_df["move2 buff"] = None 
new_df["move3 buff"] = None 
new_df["move4 buff"] = None 

new_df["move1 buff p"] = None 
new_df["move2 buff p"] = None 
new_df["move3 buff p"] = None 
new_df["move4 buff p"] = None 

In [5]:
cols = [
    "id_output",
    "move1 name",
    "move1",
    "move1_level",
    "move1 buff",
    "move1 buff p",
    "move2 name",
    "move2",
    "move2_level",
    "move2 buff",
    "move2 buff p",
    "move3 name",
    "move3",
    "move3_level",
    "move3 buff",
    "move3 buff p",
    "move4 name",
    "move4",
    "move4_level",
    "move4 buff",
    "move4 buff p",
]

filtered_df = new_df[cols]

In [6]:
filtered_df.merge(result, left_on="id_output", right_on="id_output", how="left").to_csv("Tcg game - with moves.csv", index=False)
